# 교시별 실습 Notebook

d1-p07-08

수업 페이지의 순서대로 실행합니다. 정답 블록과 선택 웹 자동화는 설명을 읽고 별도로 실행하세요.

In [ ]:
from pathlib import Path
import os

candidates = [Path.cwd(), *Path.cwd().parents]
course_root = next((p for p in candidates if (p / "raw/기존강사 수업자료").is_dir()), None)
if course_root is None:
    raise FileNotFoundError("수업 자료를 풀고 raw 폴더가 있는 위치에서 Notebook을 여세요.")
os.chdir(course_root)
print("수업 자료 폴더를 확인했습니다.")

# 요약

제공된 Titanic 1,309명 자료에서 생존 여부와 결측치를 확인하고 성별·객실 등급·나이대별 생존율을 계산합니다. 7~8교시는 같은 자료의 탐색 과정이므로 한 페이지로 묶었습니다.

## 수업 위치와 목표

**2026년 10월 6일 · 1일차 7~8교시 · 1차 초안.** 공식 「Titanic 데이터 탐색 및 기초 분석」 2시간 전체를 이 두 교시에 배치하는 제안입니다. 공식 안내문은 교시별 배분을 따로 정하지 않았습니다.

실제 파일의 컬럼·행 수와 Target 정의를 확인하고, 생존율을 분모·생존자 수와 함께 읽는 것이 목표입니다. 그룹 차이를 원인으로 단정하지 않고 관찰된 연관성으로 설명합니다.

## 교시별 진행

| 교시 | 진행 순서 | 확인할 결과 |
| --- | --- | --- |
| 7교시 | 실제 파일 열기 → 컬럼 정의 → 결측치 → Target 분포·전체 생존율 | 1309행·14열, 500/1309 약 38.20% |
| 8교시 | 성별·객실 등급 그룹 → 나이 구간 → 두 조건 결합 → 한 문장 보고 | 인원·생존자·생존율과 결측 기준을 함께 제시 |

새 Notebook `03-titanic-eda.ipynb`를 수업 프로젝트 폴더에 저장합니다. 파일은 프로젝트 기준 `raw/기존강사 수업자료/titanic.xlsx`에 있습니다. `.ipynb`를 다른 폴더에서 열었다면 `Path.cwd()`를 확인하고 `DATA`를 실제 받은 자료 폴더로 변경하세요. Excel 파일 읽기는 openpyxl이 필요합니다.

## 실습 준비

제공된 `titanic.xlsx`를 사용합니다. 이 파일은 1,309행·14열이며 컬럼 이름이 소문자입니다. 다른 Titanic 예제의 891행이나 `Survived`라는 대문자 열과 혼동하지 않습니다. Notebook의 현재 폴더에 `raw` 폴더가 보이도록 자료를 준비합니다.

## 1. Titanic 데이터 구조 및 컬럼 의미 이해

In [ ]:
from pathlib import Path
import pandas as pd

DATA = Path("raw/기존강사 수업자료")
path = DATA / "titanic.xlsx"
if not path.is_file():
    raise FileNotFoundError(f"자료 위치 확인: {path.resolve()}")
titanic = pd.read_excel(path)
print("행·열:", titanic.shape)
print("열:", titanic.columns.tolist())
print(titanic[["pclass", "survived", "sex", "age", "fare"]].head())
titanic.info()

| 열 | 의미 | 분석에서 확인할 점 |
| --- | --- | --- |
| survived | 생존 여부: 0 사망, 1 생존 | 이 단원의 Target |
| pclass | 객실 등급: 1·2·3 | 숫자지만 등급 범주 |
| sex | 성별 | 그룹별 생존율 비교 |
| age | 나이 | 결측 263개 |
| sibsp, parch | 동반 형제·배우자 / 부모·자녀 수 | 가족 구성 정보 |
| fare, embarked | 운임 / 탑승 항구 | fare 결측 1개, embarked 결측 2개 |
| name, ticket, cabin | 이름·티켓·객실 | 식별 정보나 결측이 많은 열 |
| boat, body, home.dest | 구명정·시신 번호·목적지 | 예측 시점에 알 수 있는지 따져야 함 |

`boat`와 `body`는 사고 이후의 정보를 담을 수 있습니다. 5단원 생존 예측의 입력에서 제외합니다. 이번 단원은 관찰 자료의 탐색이며, 아직 모델을 학습하지 않습니다.

**직접 실습:** `isna().sum()`을 사용해 age·fare·embarked 결측치 개수를 확인합니다. 나이가 없는 승객을 0세로 취급하지 않습니다.

## 2. 생존 여부(Target) 기준 데이터 특성 파악

0·1 값의 평균은 1인 행의 비율입니다. 생존율을 계산하기 전에 Target의 종류와 결측치를 확인합니다.

In [ ]:
print(titanic["survived"].value_counts(dropna=False).sort_index())
assert titanic["survived"].notna().all()
assert set(titanic["survived"].unique()) == {0, 1}
print(f"전체 생존율: {titanic['survived'].mean():.2%}")
target_summary = titanic.groupby("survived")[["age", "fare"]].agg(["count", "mean", "median"])
print(target_summary.round(2))

사망 809명, 생존 500명, 전체 생존율 약 38.20%입니다. age의 `count`는 그 그룹에서 나이가 알려진 사람 수이고 그룹 전체 인원과 다를 수 있습니다. 평균과 중앙값은 서로 다른 요약이며 극단값의 영향을 함께 봅니다.

**직접 실습:** 생존·사망 그룹의 운임 평균과 중앙값을 비교합니다. 평균만으로 “모든 생존자가 높은 운임을 냈다”고 말할 수 있는지 설명합니다.

## 3. 성별, 객실 등급, 나이대별 생존율 분석

비율만 비교하지 않고 각 비율의 분모인 인원을 함께 표시합니다. 아래 그룹 표의 `count`는 Target이 있는 행 수, `sum`은 생존자 수, `mean`은 생존율입니다.

In [ ]:
sex_summary = titanic.groupby("sex")["survived"].agg(["count", "sum", "mean"])
class_summary = titanic.groupby("pclass")["survived"].agg(["count", "sum", "mean"])
print("성별\n", sex_summary.round(4))
print("객실 등급별\n", class_summary.round(4))
eda = titanic.copy()
eda["age_group"] = pd.cut(
    eda["age"], bins=[0, 18, 35, 60, float("inf")],
    labels=["0~17", "18~34", "35~59", "60+"], right=False,
)
age_summary = eda.groupby("age_group", observed=True)["survived"].agg(["count", "sum", "mean"])
print("나이대별\n", age_summary.round(4))
print("나이 미상:", eda["age"].isna().sum())
both_summary = titanic.groupby(["sex", "pclass"])["survived"].agg(["count", "mean"])
print("성별과 등급을 함께 보기\n", both_summary.round(4))

여성 466명의 생존율 약 72.75%, 남성 843명의 생존율 약 19.10%입니다. 나이대 표에서는 age 결측 263명을 제외했으므로 분모의 합계는 1,046입니다. `right=False`를 사용해 18세는 `18~34` 그룹에 들어갑니다.

그룹 차이는 자료에서 관찰한 연관성입니다. 성별·객실 등급 등의 조건이 함께 달라질 수 있으므로 단일 변수의 효과나 원인으로 단정하지 않습니다.

**직접 실습:** 나이 구간을 `[0, 20, 40, 60, inf]`로 바꿉니다. 표 제목과 라벨도 바꾸고, 구간 변경 후 인원과 생존율이 달라지는지 확인합니다. 남성 1등급과 여성 3등급의 인원·생존율도 비교합니다.

## 평균이 비율이 되는 이유

생존 여부가 `[1, 0, 1, 0, 0]`이면 합계는 생존자 2명, 개수는 전체 5명입니다. 평균은 `2 / 5 = 0.4`, 즉 40%입니다. 다른 숫자가 섞이거나 1의 뜻이 생존이 아니면 이 해석을 그대로 사용할 수 없습니다.

In [ ]:
toy = pd.Series([1, 0, 1, 0, 0])
print("인원:", toy.count())
print("생존자:", toy.sum())
print(f"생존율: {toy.mean():.0%}")

`groupby`는 그룹별로 같은 계산을 반복합니다. `count`는 결측을 제외한 값의 개수, `sum`은 합계, `mean`은 평균입니다. 이번 survived는 결측이 없으므로 `count`가 승객 수와 같습니다. `age`의 count는 나이를 아는 사람 수라서 전체 승객 수와 다릅니다.

## 직접 실습 — 결측과 분모 확인

age·fare·embarked의 결측 개수를 출력합니다. 전체 인원, 나이 알려진 인원, 나이 미상 인원을 함께 출력한 뒤 두 인원 합이 전체와 같은지 확인하세요.

<details>
<summary>정답 코드와 수치</summary>

```python
print(titanic[["age", "fare", "embarked"]].isna().sum())
known_age = titanic["age"].notna().sum()
unknown_age = titanic["age"].isna().sum()
print("전체:", len(titanic))
print("나이 알려짐:", known_age)
print("나이 미상:", unknown_age)
print("인원 일치:", known_age + unknown_age == len(titanic))
```

age 263개, fare 1개, embarked 2개가 비어 있습니다. 나이 알려진 사람은 1046명, 나이 미상은 263명, 합은 1309명입니다. 나이 미상 승객을 0세로 채우면 아동 그룹의 분모와 생존율이 왜곡됩니다.

</details>

## 직접 실습 — 나이 구간을 바꾸기

나이 구간을 `[0, 20, 40, 60, inf]`로 바꾸어 `0~19`, `20~39`, `40~59`, `60+` 그룹을 만듭니다. `right=False`로 경계를 지정하고 인원·생존자·생존율을 표로 만드세요. 새 표의 인원 합은 여전히 1046명이어야 합니다.

<details>
<summary>정답과 경계 해석</summary>

```python
eda20 = titanic.copy()
eda20["age_group"] = pd.cut(
    eda20["age"], bins=[0, 20, 40, 60, float("inf")],
    labels=["0~19", "20~39", "40~59", "60+"], right=False,
)
summary20 = eda20.groupby("age_group", observed=True)["survived"].agg(
    ["count", "sum", "mean"]
)
print(summary20.round(4))
print("구간에 들어간 인원:", summary20["count"].sum())
```

구간은 0 이상 20 미만, 20 이상 40 미만, 40 이상 60 미만, 60 이상입니다. 20세는 두 번째 그룹에 들어갑니다. 라벨은 읽기 쉽게 붙인 이름이고, 실제 경계는 bins와 right 옵션이 결정합니다. 나이는 소수로 기록된 경우도 있어 `0~19` 라벨은 20세 미만을 뜻합니다.

</details>

## 직접 실습 — 두 조건과 한 문장 보고

남성 1등급과 여성 3등급의 인원·생존자·생존율을 함께 출력합니다. “어떤 파일에서, 어떤 그룹을, 몇 명 기준으로, 어떤 비율로” 비교했는지 한 문장으로 씁니다. 결과를 원인으로 설명하는 문장은 쓰지 않습니다.

<details>
<summary>정답 코드와 보고 문장 예시</summary>

```python
both = titanic.groupby(["sex", "pclass"])["survived"].agg(
    ["count", "sum", "mean"]
)
for key in [("male", 1), ("female", 3)]:
    row = both.loc[key]
    print(key, "인원:", int(row["count"]),
          "생존:", int(row["sum"]), f"생존율: {row['mean']:.2%}")
```

수치를 코드 출력에서 확인한 뒤 다음 틀을 채웁니다.

“제공된 titanic.xlsx의 남성 1등급과 여성 3등급을 비교했을 때, 각 그룹의 인원은 ___명·___명이고 관찰된 생존율은 ___%·___%였다.”

이어 “성별과 등급 이외의 조건도 함께 달라질 수 있으므로 이 표만으로 원인을 단정하지 않는다”라고 적습니다. 몇 퍼센트인지 기억해서 적기보다 실행 결과로 보고서를 채우는 습관을 익힙니다.

</details>

## 흔한 오류와 복구

| 오류·현상 | 확인과 해결 |
| --- | --- |
| 파일을 찾지 못함 | `Path.cwd()`와 `path.resolve()`를 확인합니다. 프로젝트 raw 폴더가 기준 경로에 있어야 합니다. |
| 891행 또는 Survived 컬럼이 나옴 | 다른 Titanic 데이터입니다. 이번 제공 파일은 1309행·14열, 소문자 survived입니다. |
| `KeyError: Survived` | 실제 컬럼은 survived입니다. `titanic.columns.tolist()`를 먼저 확인합니다. |
| 그룹 인원 합이 1046명 | 나이 미상 263명이 나이 구간에서 빠졌습니다. age 결측 개수를 함께 표시합니다. |
| 생존율을 0.7275%로 씀 | `0.7275`는 비율이고 약 72.75%입니다. `:.2%` 서식을 사용합니다. |
| 생존자의 나이 count가 그룹 전체 인원과 다름 | count는 해당 열에서 결측이 없는 개수입니다. survived count와 age count를 구분합니다. |
| boat·body로 예측하려 함 | 사고 이후 정보일 수 있으므로 이후 예측 입력에서 제외해야 합니다. |

## 완료 체크

- [ ] 수업 폴더에서 uv로 Notebook을 열고 Kernel의 `sys.executable`이 수업 `.venv`인지 확인했습니다. [환경 안내](surl:14892)
- [ ] 실습을 저장한 뒤 Kernel을 재시작하고 선행 페이지의 준비 셀부터 위에서 아래로 다시 실행했습니다. 웹 수집은 저장한 자료와 성공·실패 기록을 구분했습니다.

- [ ] 실제 제공 파일에서 1309행·14열과 소문자 컬럼을 확인했습니다.
- [ ] survived가 0·1이고 결측이 없는지 확인했습니다.
- [ ] 사망 809명·생존 500명과 전체 생존율 약 38.20%를 확인했습니다.
- [ ] 그룹마다 인원·생존자 수·생존율을 함께 표시했습니다.
- [ ] 나이 미상 263명과 구간 경계를 설명했습니다.
- [ ] 관찰된 그룹 차이를 원인으로 단정하지 않았습니다.

## 원자료와 이어서 보기

공식 안내문 3쪽의 Titanic 탐색 2시간 항목을 설명하며, 기존 day1-2-분석.ipynb와 3장 PPT를 보조 자료로 사용합니다. 2일차에는 같은 자료를 그래프로 표현하고 모델 입력을 준비합니다.

[수업 전체 안내](surl:14880) · [기존 단원: Titanic 데이터 탐색 및 기초 분석](surl:14884) · [다음 단원: Titanic 데이터 시각화 분석](surl:14885)

<!-- period-navigation:start -->
## 교시별 이동

[교시별 전체 안내](surl:14891) · [공식 단원별 안내](surl:14880) · [이전: 10월 6일 6교시 — BeautifulSoup과 Selenium 웹 수집](surl:14895) · [다음: 10월 7일 1~2교시 — Titanic 시각화와 그래프](surl:14897)
<!-- period-navigation:end -->